Module 3: Hệ Khuyến Nghị Cá Nhân Hóa
Phương pháp: ALS (Alternating Least Squares) - Implicit Collaborative Filtering  
Dữ liệu: Lịch sử nghe nhạc (user_id, track_name, artist_name, timestamp)  
Đặc điểm: Batch training từng file, lưu model checkpoint sau mỗi batch


1. Cài đặt thư viện

In [1]:
!pip install scikit-learn scipy pandas numpy tqdm joblib implicit -q
import warnings
warnings.filterwarnings('ignore')
print('Cài đặt hoàn tất')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cài đặt hoàn tất


2. Import & Cấu hình

In [4]:
import os, glob, gc, pickle, json, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm
from collections import defaultdict
from implicit.als import AlternatingLeastSquares
import implicit.cpu.als as cpu_als
import joblib

CONFIG = {
    'data_dir'        : '/kaggle/input/datasets/js042710/second3t1k',
    'output_dir'      : '/kaggle/working/model',
    'checkpoint_dir'  : '/kaggle/working/checkpoints',
    # ALS params
    'factors'         : 256,
    'iterations'      : 30,
    'regularization'  : 0.05,
    'alpha'           : 15,
    'use_gpu'         : True,   # GPU để train nhanh
    'num_threads'     : 4,      # fallback nếu không có GPU
    # Data
    'min_interactions': 20,
    'item_key'        : 'track_artist',
    'chunk_size'      : 500_000,
}

# Kiểm tra GPU thực sự có sẵn không
try:
    import implicit.gpu
    HAS_GPU = implicit.gpu.HAS_CUDA
except:
    HAS_GPU = False

CONFIG['use_gpu'] = HAS_GPU
print(f'GPU available: {HAS_GPU}')
print(f'→ Sẽ train bằng: {"GPU" if HAS_GPU else "CPU"}')

os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
print('Config loaded')
print(json.dumps({k:v for k,v in CONFIG.items() if 'dir' not in k}, indent=2))


GPU available: True
→ Sẽ train bằng: GPU
Config loaded
{
  "factors": 256,
  "iterations": 30,
  "regularization": 0.05,
  "alpha": 15,
  "use_gpu": true,
  "num_threads": 4,
  "min_interactions": 20,
  "item_key": "track_artist",
  "chunk_size": 500000
}


3. Khám phá dữ liệu

In [5]:
# Cập nhật code: Tìm kiếm đệ quy (quét cả thư mục gốc và thư mục con)
all_files = sorted(glob.glob(os.path.join(CONFIG['data_dir'], '**', '*.csv'), recursive=True))
all_files += sorted(glob.glob(os.path.join(CONFIG['data_dir'], '**', '*.xlsx'), recursive=True))

print(f'Tổng số file: {len(all_files)}')
for f in all_files[:5]:
    print(' -', f) # In ra đường dẫn đầy đủ để bạn dễ kiểm tra

# Preview file đầu tiên
if len(all_files) > 0:
    df_sample = pd.read_csv(all_files[0], nrows=5)
    print('\nSchema:')
    print(df_sample.dtypes)
    print('\nSample:')
    display(df_sample)
else:
    print("❌ Vẫn không tìm thấy file nào. Hãy kiểm tra lại thư mục!")

Tổng số file: 62
 - /kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA/clean_1.2.2026.listens.csv
 - /kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA/clean_10.1.2026.listens.csv
 - /kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA/clean_11.1.2026.listens.csv
 - /kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA/clean_12.1.2026.listens.csv
 - /kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA/clean_13.1.2026.listens.csv

Schema:
user_id            int64
timestamp         object
recording_msid    object
track_name        object
artist_name       object
release_name      object
dtype: object

Sample:


,user_id,timestamp,recording_msid,track_name,artist_name,release_name
0,1,2026-01-31 12:57:58,2d39940b-f575-42a7-b636-e3ab6880bbca,Auxin,Cell,Onwards System
1,1,2026-01-31 13:04:37,230a1f84-609d-43d1-a50e-bff4a34aff0c,Figment,Cell,Onwards System
2,1,2026-01-31 13:11:07,025df4fb-5bc4-4b34-814d-ec6aa6fe08aa,Geiger,Cell,Onwards System
3,5,2026-01-31 08:04:43,11fa2a40-66e4-40a0-bd4f-3a06c9848644,You Know How We Do (Camilo Dos Santos & Crafts...,Ice Cube,Chapter Eighteen Edits
4,5,2026-01-31 08:36:21,8d4f83c7-a6fc-497d-9be8-4ff457656c9d,House Down,Even Funkier,Chapter Eighteen Edits


4. Xây dựng Index (User & Item)

In [6]:
class InteractionAccumulator:
    def __init__(self):
        self.user2idx     = {}
        self.item2idx     = {}
        self.idx2user     = []
        self.idx2item     = []
        self.interactions = defaultdict(float)
        self.n_rows_processed = 0

    def _get_or_create(self, mapping, reverse, key):
        if key not in mapping:
            mapping[key] = len(reverse)
            reverse.append(key)
        return mapping[key]

    def add_dataframe(self, df: pd.DataFrame):
        cols = ['user_id','track_name','artist_name']
        has_ts = 'timestamp' in df.columns
        if has_ts: cols.append('timestamp')
        df = df[cols].dropna()
        df['item_key'] = df['track_name'].str.strip() + '|||' + df['artist_name'].str.strip()
        if has_ts:
            df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            df = df.dropna(subset=['timestamp'])
            local_max = df['timestamp'].max()
            days_ago = (local_max - df['timestamp']).dt.days.clip(lower=0)
            df['weight'] = np.exp(-days_ago / 60.0)  # half-life 60 ngày
        else:
            df['weight'] = 1.0
        for row in df.itertuples(index=False):
            uid = self._get_or_create(self.user2idx, self.idx2user, str(row.user_id))
            iid = self._get_or_create(self.item2idx, self.idx2item, row.item_key)
            self.interactions[(uid, iid)] += row.weight
        self.n_rows_processed += len(df)

    def build_matrix(self, min_interactions=20):
        print(f' Building matrix từ {len(self.interactions):,} cặp tương tác...')
        rows, cols, data = [], [], []
        for (uid, iid), cnt in self.interactions.items():
            rows.append(uid); cols.append(iid); data.append(cnt)
        mat = sp.csr_matrix(
            (data,(rows,cols)),
            shape=(len(self.idx2user), len(self.idx2item)),
            dtype=np.float32
        )
        if min_interactions > 1:
            mat_bin = (mat > 0).astype(np.float32)
            ic_cnt  = np.asarray(mat_bin.sum(axis=0)).flatten()
            uc_cnt  = np.asarray(mat_bin.sum(axis=1)).flatten()
            vi = ic_cnt >= min_interactions
            vu = uc_cnt >= min_interactions
            mat = mat[vu][:, vi]
            self.idx2user = [u for u,v in zip(self.idx2user, vu) if v]
            self.idx2item = [i for i,v in zip(self.idx2item, vi) if v]
            self.user2idx = {u:i for i,u in enumerate(self.idx2user)}
            self.item2idx = {i:j for j,i in enumerate(self.idx2item)}
            print(f'   Sau lọc (min={min_interactions}): {mat.shape[0]:,} users, {mat.shape[1]:,} items')
        density = mat.nnz / (mat.shape[0]*mat.shape[1]) * 100
        print(f'   Ma trận: {mat.shape[0]:,} × {mat.shape[1]:,} | Density: {density:.4f}%')
        return mat

accumulator = InteractionAccumulator()
print('Khởi tạo InteractionAccumulator')


Khởi tạo InteractionAccumulator


5. Load dữ liệu theo Batch

In [7]:
CHECKPOINT_INDEX_PATH = os.path.join(CONFIG['checkpoint_dir'], 'accumulator_checkpoint.pkl')
PROGRESS_PATH = os.path.join(CONFIG['checkpoint_dir'], 'progress.json')

def save_accumulator_checkpoint(acc, processed_files):
    """Lưu checkpoint accumulator."""
    state = {
        'user2idx': acc.user2idx,
        'item2idx': acc.item2idx,
        'idx2user': acc.idx2user,
        'idx2item': acc.idx2item,
        'interactions': dict(acc.interactions),
        'n_rows_processed': acc.n_rows_processed,
    }
    joblib.dump(state, CHECKPOINT_INDEX_PATH, compress=3)
    with open(PROGRESS_PATH, 'w') as f:
        json.dump({'processed_files': processed_files}, f)
    print(f'   Checkpoint lưu: {len(processed_files)} files')

def load_accumulator_checkpoint():
    """Khôi phục từ checkpoint nếu tồn tại."""
    if not os.path.exists(CHECKPOINT_INDEX_PATH):
        return InteractionAccumulator(), []
    state = joblib.load(CHECKPOINT_INDEX_PATH)
    acc = InteractionAccumulator()
    acc.user2idx = state['user2idx']
    acc.item2idx = state['item2idx']
    acc.idx2user = state['idx2user']
    acc.idx2item = state['idx2item']
    acc.interactions = defaultdict(int, state['interactions'])
    acc.n_rows_processed = state['n_rows_processed']
    with open(PROGRESS_PATH) as f:
        progress = json.load(f)
    print(f'Khôi phục từ checkpoint: {len(progress["processed_files"])} files đã xử lý')
    return acc, progress['processed_files']

# ─── LOAD DỮ LIỆU ──────────────────────────────────────────
RESUME_FROM_CHECKPOINT = True  # ← False để train lại từ đầu

if RESUME_FROM_CHECKPOINT:
    accumulator, processed_files = load_accumulator_checkpoint()
else:
    accumulator, processed_files = InteractionAccumulator(), []

remaining_files = [f for f in all_files if f not in processed_files]
print(f'Còn {len(remaining_files)} files cần xử lý')

Còn 62 files cần xử lý


In [9]:
SAVE_EVERY_N_FILES = 15   # Lưu checkpoint sau mỗi N file

t0 = time.time()
for i, fpath in enumerate(tqdm(remaining_files, desc='Loading files')):
    fname = os.path.basename(fpath)
    try:
        if fpath.endswith('.xlsx'):
            df = pd.read_excel(fpath, usecols=['user_id','track_name','artist_name'])
            accumulator.add_dataframe(df)
        else:
            # Đọc theo chunk để tiết kiệm RAM
            for chunk in pd.read_csv(
                fpath,
                usecols=['user_id','track_name','artist_name'],
                chunksize=CONFIG['chunk_size'],
                low_memory=False
            ):
                accumulator.add_dataframe(chunk)
                del chunk
                gc.collect()

        processed_files.append(fpath)

        # Lưu checkpoint định kỳ
        if (i + 1) % SAVE_EVERY_N_FILES == 0:
            save_accumulator_checkpoint(accumulator, processed_files)

    except Exception as e:
        print(f'Lỗi file {fname}: {e}')
        continue

# Lưu lần cuối
save_accumulator_checkpoint(accumulator, processed_files)

elapsed = time.time() - t0
print(f'\nXử lý xong {len(processed_files)} files trong {elapsed/60:.1f} phút')
print(f'   Tổng số dòng: {accumulator.n_rows_processed:,}')
print(f'   Users: {len(accumulator.idx2user):,}')
print(f'   Items: {len(accumulator.idx2item):,}')

Loading files:  24%|██▍       | 15/62 [07:51<58:11, 74.29s/it]

   Checkpoint lưu: 35 files


Loading files:  48%|████▊     | 30/62 [16:59<48:16, 90.52s/it]

   Checkpoint lưu: 50 files


Loading files:  73%|███████▎  | 45/62 [29:39<35:39, 125.86s/it]

   Checkpoint lưu: 65 files


Loading files:  97%|█████████▋| 60/62 [42:45<05:01, 150.85s/it]

   Checkpoint lưu: 80 files


Loading files: 100%|██████████| 62/62 [46:03<00:00, 44.57s/it] 


   Checkpoint lưu: 82 files

Xử lý xong 82 files trong 52.8 phút
   Tổng số dòng: 222,925,786
   Users: 28,775
   Items: 13,780,862


6. Xây dựng Sparse Matrix & Train ALS

In [10]:
# Xây dựng user-item matrix
user_item_matrix = accumulator.build_matrix(min_interactions=CONFIG['min_interactions'])

# ALS yêu cầu item × user cho training
item_user_matrix = user_item_matrix.T.tocsr()

print(f'\nSẵn sàng train ALS')
print(f'   item×user matrix: {item_user_matrix.shape}')

 Building matrix từ 45,421,595 cặp tương tác...
   Sau lọc (min=20): 23,951 users, 318,070 items
   Ma trận: 23,951 × 318,070 | Density: 0.2394%

Sẵn sàng train ALS
   item×user matrix: (318070, 23951)


In [11]:
# ─── TRAIN/TEST SPLIT (user-based) ────────────────────────
# Dùng leave-one-out theo user thay vì random split
# → đảm bảo mỗi user đều có trong cả train lẫn test
def user_based_split(user_item, test_ratio=0.2, min_items=5):
    train = user_item.tolil().astype(np.float32)
    test  = sp.lil_matrix(user_item.shape, dtype=np.float32)
    rng   = np.random.default_rng(42)
    for u in range(user_item.shape[0]):
        items = user_item[u].indices
        if len(items) < min_items:
            continue
        n_test = max(1, int(len(items) * test_ratio))
        test_items = rng.choice(items, n_test, replace=False)
        for it in test_items:
            test[u, it]  = user_item[u, it]
            train[u, it] = 0
    return train.tocsr(), test.tocsr()

print('Đang tạo user-based split...')
train_matrix, test_matrix = user_based_split(user_item_matrix, test_ratio=0.2)
train_item_user = train_matrix.T.tocsr()
print(f'Train: {train_matrix.nnz:,} interactions')
print(f'Test:  {test_matrix.nnz:,} interactions')
print(f'Users có trong test: {(test_matrix.sum(axis=1)>0).sum():,}')


Đang tạo user-based split...
Train: 14,601,583 interactions
Test:  3,638,520 interactions
Users có trong test: 22,954


In [12]:
# ─── TRAIN ALS MODEL ───────────────────────────────────────
import implicit.cpu.als as cpu_als

MODEL_CHECKPOINT = os.path.join(CONFIG['checkpoint_dir'], 'als_model_latest.pkl')
RESUME_MODEL = False

if RESUME_MODEL and os.path.exists(MODEL_CHECKPOINT):
    print('Load model từ checkpoint...')
    model = joblib.load(MODEL_CHECKPOINT)
    print('Model loaded')
else:
    print(f'Train ALS trên {"GPU" if CONFIG["use_gpu"] else "CPU"}...')
    model = AlternatingLeastSquares(
        factors                 = CONFIG['factors'],
        iterations              = CONFIG['iterations'],
        regularization          = CONFIG['regularization'],
        alpha                   = CONFIG['alpha'],
        use_gpu                 = CONFIG['use_gpu'],
        num_threads             = CONFIG.get('num_threads', 4),
        calculate_training_loss = True,
        random_state            = 42,
    )
    t0 = time.time()
    model.fit(train_item_user)
    print(f'Train xong trong {(time.time()-t0)/60:.1f} phút')

    
    if CONFIG['use_gpu']:
        print('Converting GPU → CPU...')
        
        item_f = model.item_factors.to_numpy()
        user_f = model.user_factors.to_numpy()
        cpu_model = cpu_als.AlternatingLeastSquares(
            factors=CONFIG['factors'], iterations=0,
            regularization=CONFIG['regularization'],
        )
        cpu_model.item_factors = user_f   
        cpu_model.user_factors = item_f   
        model = cpu_model
        print('Convert xong')

    joblib.dump(model, MODEL_CHECKPOINT, compress=3)
    print(f'Lưu CPU model: {MODEL_CHECKPOINT}')

Train ALS trên GPU...


  0%|          | 0/30 [00:00<?, ?it/s]

Train xong trong 0.6 phút
Converting GPU → CPU...
Convert xong
Lưu CPU model: /kaggle/working/checkpoints/als_model_latest.pkl


In [13]:
print(f'item_factors shape: {model.item_factors.shape}')
print(f'user_factors shape: {model.user_factors.shape}')
print(f'train_matrix shape: {train_matrix.shape}')

item_factors shape: (318070, 256)
user_factors shape: (23951, 256)
train_matrix shape: (23951, 318070)


7. Đánh giá Model

In [14]:
# ─── ĐÁNH GIÁ MODEL
import numpy as np

def evaluate_model(model, train_mat, test_mat, K=10, n_users=2000):
    """
    Tính MAP@K, Precision@K, Recall@K thủ công.
    Tương thích với mọi phiên bản scipy/implicit.
    """
    rng = np.random.default_rng(42)
    n_eval = min(n_users, test_mat.shape[0])
    test_users = rng.choice(test_mat.shape[0], n_eval, replace=False)

    ap_list, prec_list, rec_list = [], [], []

    for u in test_users:
        actual = set(test_mat[u].indices)
        if not actual:
            continue

        rec_ids, _ = model.recommend(
            u, train_mat[u], N=K,
            filter_already_liked_items=True
        )
        rec_ids = list(rec_ids)

        # Precision@K
        hits = len(set(rec_ids) & actual)
        prec_list.append(hits / K)

        # Recall@K
        rec_list.append(hits / len(actual))

        # AP@K
        ap, n_hits = 0.0, 0
        for rank, item in enumerate(rec_ids, 1):
            if item in actual:
                n_hits += 1
                ap += n_hits / rank
        ap_list.append(ap / min(len(actual), K))

    print(f'Kết quả đánh giá (n_users={n_eval}, K={K}):')
    print(f'   MAP@{K}       : {np.mean(ap_list):.4f}')
    print(f'   Precision@{K} : {np.mean(prec_list):.4f}')
    print(f'   Recall@{K}    : {np.mean(rec_list):.4f}')
    return np.mean(ap_list), np.mean(prec_list), np.mean(rec_list)

map_at_10, p10, r10 = evaluate_model(model, train_matrix, test_matrix, K=10)
_, p20, r20          = evaluate_model(model, train_matrix, test_matrix, K=20)

Kết quả đánh giá (n_users=2000, K=10):
   MAP@10       : 0.1391
   Precision@10 : 0.1911
   Recall@10    : 0.0492
Kết quả đánh giá (n_users=2000, K=20):
   MAP@20       : 0.1062
   Precision@20 : 0.1557
   Recall@20    : 0.0693


8. Lưu Model Hoàn Chỉnh

In [16]:
FINAL_MODEL_DIR = CONFIG['output_dir']

# 1. Lưu ALS model
joblib.dump(model, os.path.join(FINAL_MODEL_DIR, 'als_model.pkl'), compress=3)

# 2. Lưu index mappings
joblib.dump({
    'user2idx': accumulator.user2idx,
    'item2idx': accumulator.item2idx,
    'idx2user': accumulator.idx2user,
    'idx2item': accumulator.idx2item,
}, os.path.join(FINAL_MODEL_DIR, 'index_mappings.pkl'), compress=3)

# 3. Lưu user-item sparse matrix (để reuse)
sp.save_npz(os.path.join(FINAL_MODEL_DIR, 'user_item_matrix.npz'), user_item_matrix)

# 4. Lưu item metadata (track_name + artist_name)
item_meta = pd.DataFrame([
    {'item_idx': i, 'track_name': k.split('|||')[0], 'artist_name': k.split('|||')[1]}
    for i, k in enumerate(accumulator.idx2item)
])
item_meta.to_parquet(os.path.join(FINAL_MODEL_DIR, 'item_metadata.parquet'), index=False)

# 5. Lưu config
with open(os.path.join(FINAL_MODEL_DIR, 'config.json'), 'w') as f:
    json.dump({
        **CONFIG,
        'n_users': len(accumulator.idx2user),
        'n_items': len(accumulator.idx2item),
        'map_at_10': float(map_at_10),
        'precision_at_10': float(p10),
    }, f, indent=2)

print('Đã lưu đầy đủ:')
for f in os.listdir(FINAL_MODEL_DIR):
    fpath = os.path.join(FINAL_MODEL_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'   {f:40s} {size_mb:8.2f} MB')

Đã lưu đầy đủ:
   item_metadata.parquet                        7.49 MB
   als_model.pkl                              326.62 MB
   index_mappings.pkl                           6.60 MB
   config.json                                  0.00 MB
   user_item_matrix.npz                        46.50 MB


9. Hàm Khuyến Nghị

In [17]:
class MusicRecommender:
    def __init__(self, model_dir: str):
        self.model     = joblib.load(os.path.join(model_dir, 'als_model.pkl'))
        mappings       = joblib.load(os.path.join(model_dir, 'index_mappings.pkl'))
        self.user2idx  = mappings['user2idx']
        self.item2idx  = mappings['item2idx']
        self.idx2user  = mappings['idx2user']
        self.idx2item  = mappings['idx2item']
        self.user_item = sp.load_npz(os.path.join(model_dir, 'user_item_matrix.npz'))
        self.item_meta = pd.read_parquet(os.path.join(model_dir, 'item_metadata.parquet'))
        print(f'Model loaded | {len(self.idx2user):,} users | {len(self.idx2item):,} items')

    def _item_idx_to_df(self, item_ids, scores):
        rows = []
        for idx, score in zip(item_ids, scores):
            key = self.idx2item[idx]
            track, artist = key.split('|||')
            rows.append({'track_name': track, 'artist_name': artist, 'score': round(float(score),4)})
        return pd.DataFrame(rows)

    def recommend_for_user(self, user_id, n=10, filter_listened=True):
        uid_str = str(user_id)
        if uid_str not in self.user2idx:
            print(f'User {user_id} chưa có → trả về bài phổ biến.')
            return self.popular_items(n)
        uid = self.user2idx[uid_str]
        rec_ids, scores = self.model.recommend(
            uid, self.user_item[uid], N=n,
            filter_already_liked_items=filter_listened
        )
        df = self._item_idx_to_df(list(rec_ids), list(scores))
        df.index = range(1, len(df)+1)
        return df

    def similar_tracks(self, track_name, artist_name, n=10):
        key = f'{track_name.strip()}|||{artist_name.strip()}'
        if key not in self.item2idx:
            print(f'Không tìm thấy: "{track_name}" - {artist_name}')
            return pd.DataFrame()
        item_idx = self.item2idx[key]
        sim_ids, scores = self.model.similar_items(item_idx, N=n+1)
        filtered = [(i,s) for i,s in zip(list(sim_ids),list(scores)) if i != item_idx][:n]
        if not filtered: return pd.DataFrame()
        ids, sims = zip(*filtered)
        df = self._item_idx_to_df(ids, sims)
        df.index = range(1, len(df)+1)
        return df

    def similar_users(self, user_id, n=10):
        uid_str = str(user_id)
        if uid_str not in self.user2idx: return pd.DataFrame()
        uid = self.user2idx[uid_str]
        sim_ids, scores = self.model.similar_users(uid, N=n+1)
        filtered = [(i,s) for i,s in zip(list(sim_ids),list(scores)) if i != uid][:n]
        if not filtered: return pd.DataFrame()
        ids, sims = zip(*filtered)
        df = pd.DataFrame({
            'user_id'   : [self.idx2user[i] for i in ids],
            'similarity': [round(float(s),4) for s in sims]
        })
        df.index = range(1, len(df)+1)
        return df

    def popular_items(self, n=10):
        item_counts = np.asarray(self.user_item.sum(axis=0)).flatten()
        top_ids = np.argsort(item_counts)[::-1][:n]
        df = self._item_idx_to_df(top_ids, item_counts[top_ids])
        df.rename(columns={'score':'play_count'}, inplace=True)
        df.index = range(1, len(df)+1)
        return df

    def user_history(self, user_id, n=20):
        uid_str = str(user_id)
        if uid_str not in self.user2idx: return pd.DataFrame()
        uid = self.user2idx[uid_str]
        row = self.user_item[uid]
        top_n = np.argsort(row.data)[::-1][:n]
        df = self._item_idx_to_df(row.indices[top_n], row.data[top_n])
        df.rename(columns={'score':'weighted_plays'}, inplace=True)
        df.index = range(1, len(df)+1)
        return df

print('Class MusicRecommender định nghĩa xong')


Class MusicRecommender định nghĩa xong


10. Demo & Test

In [18]:
# Load recommender
rec = MusicRecommender(CONFIG['output_dir'])

# ─── Test với user thực tế ─────────────────────────────────
TEST_USER_ID = 1   # ← thay bằng user_id bất kỳ

print(f'\nLịch sử nghe của User {TEST_USER_ID}:')
display(rec.user_history(TEST_USER_ID, n=10))

print(f'\nTop 10 khuyến nghị cho User {TEST_USER_ID}:')
display(rec.recommend_for_user(TEST_USER_ID, n=10))

Model loaded | 23,951 users | 318,070 items

Lịch sử nghe của User 1:


,track_name,artist_name,weighted_plays
1,Seeker,Carbon Based Lifeforms,7.0
2,Fauna,Carbon Based Lifeforms,7.0
3,The Veil,Röyksopp,6.0
4,I'm There With You,Röyksopp,6.0
5,Sync2n,Carbon Based Lifeforms,5.0
6,Dandelion Pleasantries,Röyksopp,5.0
7,Interloper,Carbon Based Lifeforms,4.0
8,Denimclad Baboons,Röyksopp,3.0
9,Frog,Carbon Based Lifeforms,3.0
10,Remembering the Departed,Röyksopp,3.0



Top 10 khuyến nghị cho User 1:


,track_name,artist_name,score
1,Accede,Carbon Based Lifeforms,0.0259
2,Arecibo,Carbon Based Lifeforms,0.0258
3,Photosynthesis,Carbon Based Lifeforms,0.0245
4,Abiogenesis,Carbon Based Lifeforms,0.0241
5,Central Plains,Carbon Based Lifeforms,0.0239
6,Tensor,Carbon Based Lifeforms,0.0233
7,Gryning,Carbon Based Lifeforms,0.0230
8,Exosphere,Carbon Based Lifeforms,0.0226
9,Why Does My Heart Feel So Bad?,Moby,0.0226
10,20 Minutes,Carbon Based Lifeforms,0.0226


In [19]:
# ─── Bài hát tương tự ─────────────────────────────────────
print('Bài hát tương tự "Auxin" - Cell:')
display(rec.similar_tracks('Auxin', 'Cell', n=10))

# ─── User có gu tương tự ──────────────────────────────────
print(f'\nUsers có gu nhạc giống User {TEST_USER_ID}:')
display(rec.similar_users(TEST_USER_ID, n=5))

# ─── Cold-start: bài phổ biến nhất ────────────────────────
print('\nTop bài hát phổ biến nhất (Cold-start fallback):')
display(rec.popular_items(n=10))

Bài hát tương tự "Auxin" - Cell:
Không tìm thấy: "Auxin" - Cell


""



Users có gu nhạc giống User 1:


,user_id,similarity
1,9519,0.7272
2,70461,0.7032
3,65509,0.6147
4,8815,0.5879
5,29017,0.5857



Top bài hát phổ biến nhất (Cold-start fallback):


,track_name,artist_name,play_count
1,Mr. Brightside,The Killers,26925.0
2,In the End,Linkin Park,25891.0
3,Numb,Linkin Park,25101.0
4,Blinding Lights,The Weeknd,25100.0
5,Creep,Radiohead,22745.0
6,Seven Nation Army,The White Stripes,21554.0
7,Smells Like Teen Spirit,Nirvana,20326.0
8,Everlong,Foo Fighters,19975.0
9,Take Me Out,Franz Ferdinand,19920.0
10,Do I Wanna Know?,Arctic Monkeys,19871.0


11. Fine-tune / Update Model với dữ liệu mới:
Khi có thêm file dữ liệu mới, chạy lại cell dưới để update model mà không cần train từ đầu.

In [ ]:
def update_model_with_new_data(new_file_paths: list, model_dir: str, n_iter_update=10):
    """
    Cập nhật ALS model với dữ liệu mới.
    Tích lũy thêm vào accumulator hiện có, re-fit model.
    """
    # Load accumulator cũ
    old_acc, old_processed = load_accumulator_checkpoint()

    # Thêm dữ liệu mới
    for fpath in tqdm(new_file_paths, desc='Loading new data'):
        for chunk in pd.read_csv(
            fpath,
            usecols=['user_id','track_name','artist_name'],
            chunksize=500_000
        ):
            old_acc.add_dataframe(chunk)

    # Rebuild matrix
    new_matrix = old_acc.build_matrix(min_interactions=CONFIG['min_interactions'])

    # Re-fit model (warm start bằng cách dùng ít iterations hơn)
    new_model = AlternatingLeastSquares(
        factors=CONFIG['factors'],
        iterations=n_iter_update,
        regularization=CONFIG['regularization'],
        alpha=CONFIG['alpha'],
        use_gpu=CONFIG['use_gpu'],
        random_state=42,
    )
    new_model.fit(new_matrix.T.tocsr())

    # Lưu
    joblib.dump(new_model, os.path.join(model_dir, 'als_model.pkl'), compress=3)
    joblib.dump({
        'user2idx': old_acc.user2idx, 'item2idx': old_acc.item2idx,
        'idx2user': old_acc.idx2user, 'idx2item': old_acc.idx2item,
    }, os.path.join(model_dir, 'index_mappings.pkl'), compress=3)
    sp.save_npz(os.path.join(model_dir, 'user_item_matrix.npz'), new_matrix)
    save_accumulator_checkpoint(old_acc, old_processed + new_file_paths)

    print(f'Model đã được cập nhật với {len(new_file_paths)} file mới')
    return new_model, old_acc

# Ví dụ:
# update_model_with_new_data(['/kaggle/input/newdata/clean_1.3.2026.csv'], CONFIG['output_dir'])
print('Hàm update_model_with_new_data đã sẵn sàng')